# Neuronal: primeira sessão de pesquisa

Autor: José Henrique Targino Dias Gois.

Objetivo: distinguir instalação PyPI de desenvolvimento local e experimentar a fronteira Rust–Python.
Execute as células em ordem. Este notebook não instala pacotes nem modifica arquivos.

## Preparar o kernel

Siga ../README.md para criar os kernels **Neuronal — PyPI** e **Neuronal — Local**.
PyPI: pip install neuronal-rs==0.1.0. Local: maturin develop --locked.
Selecione o kernel no menu Kernel → Change Kernel.

As primeiras experiências funcionam nos dois; as novas APIs são locais.

In [ ]:
import sys
import platform
import importlib.metadata as metadata
import neuronal

print("Python:", sys.executable)
print("Sistema:", platform.platform())
print("Módulo:", neuronal.__file__)
print("Distribuição:", metadata.version("neuronal-rs"))
print("Novas APIs locais:", hasattr(neuronal, "sampled_span"))
for package in ("numpy", "matplotlib", "ipykernel"):
    print(package, metadata.version(package))

## Hipótese e parâmetros

Hipótese: para N amostras com passo dt, t[i] = i × dt e o último instante é (N−1) × dt.
Isso é a distância entre primeira e última amostra; não confundir com N × dt.
Não simulamos neurônios ainda.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

samples = 100
dt_ms = 0.1
times = np.asarray(neuronal.time_axis(samples, dt_ms))
expected = np.arange(samples) * dt_ms
np.testing.assert_allclose(times, expected)
print("Último instante (ms):", times[-1])

In [ ]:
fig, ax = plt.subplots()
ax.plot(np.arange(samples), times)
ax.set_xlabel("Índice da amostra")
ax.set_ylabel("Tempo (ms)")
plt.show()

## Protótipo em Python

Uma nova ideia começa numa célula. Depois de verificá-la, decidimos se merece virar API da biblioteca.

In [ ]:
def sampled_span_python(samples, dt_ms):
    if samples < 0:
        raise ValueError("samples deve ser não negativo")
    if not np.isfinite(dt_ms) or dt_ms <= 0:
        raise ValueError("dt_ms deve ser finito e positivo")
    return max(samples - 1, 0) * dt_ms

sampled_span_python(samples, dt_ms)

## Implementação Rust e interface

O mesmo cálculo agora está implementado no código local:

| Camada | Arquivo | Papel |
|---|---|---|
| Rust | crates/neuronal-core/src/lib.rs | sampled_span e validação numérica |
| PyO3 | crates/neuronal-python/src/lib.rs | #[pyfunction] e registro no módulo |
| Python | python/neuronal/__init__.py | exportação pública |
| Tipagem | python/neuronal/_native.pyi | assinatura para editores |

Após editar Rust, na raiz do projeto e no ambiente local:

```bash
cargo fmt --all
cargo check --workspace --locked
maturin develop --locked
pytest
```

Reinicie o kernel antes de executar a extensão recompilada. A versão PyPI 0.1.0 ainda não oferece sampled_span.

In [ ]:
if hasattr(neuronal, "sampled_span"):
    rust_span = neuronal.sampled_span(samples, dt_ms)
    np.testing.assert_allclose(rust_span, sampled_span_python(samples, dt_ms))
    print("Rust e Python concordam:", rust_span)
else:
    print("API ausente na versão publicada. Selecione o kernel Local após maturin develop.")

## Conveniência em Python

python/neuronal/plotting.py usa time_axis de Rust e Matplotlib de Python.
Edite, por exemplo, a cor da curva nesse arquivo. O bloco abaixo recarrega somente esse módulo Python.
Não é necessário recompilar Rust para essa alteração.

In [ ]:
import importlib

if hasattr(neuronal, "sampled_span"):
    import neuronal.plotting as plotting
    importlib.reload(plotting)
    fig, ax = plotting.plot_time_axis(samples=samples, dt_ms=dt_ms)
    plt.show()
else:
    print("neuronal.plotting está disponível no desenvolvimento local.")

## Registrar resultados

- Hipótese: ...
- Parâmetros: samples = 100, dt_ms = 0.1.
- Origem do pacote: PyPI ou local? ...
- Commit local (git rev-parse HEAD no terminal): ...
- Resultado observado: ...
- Conclusão e próximo experimento: ...

Use examples/results/ para figuras e examples/data/ para dados locais; ambos estão ignorados.
Salve este notebook sem saídas antes de commitar. Mudanças locais não alteram o PyPI.
Próxima investigação: LIF determinístico e validação contra solução analítica.